Files looking at:
- `minuteIntensitiesWide_merged`
- `minuteStepsNarrow_merged`
- `minuteStepsWide_merged`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

### Load Data

In [ ]:
# Load data
DATA_LOC = '../data/'

minute_intensities_wide = pd.read_csv(f'{DATA_LOC}minuteIntensitiesWide_merged.csv')
minute_steps_narrow     = pd.read_csv(f'{DATA_LOC}minuteStepsNarrow_merged.csv')
minute_steps_wide       = pd.read_csv(f'{DATA_LOC}minuteStepsWide_merged.csv')

### Missing Values

In [ ]:
# Find the count of missing values
print('Missing values')

na_counts = minute_intensities_wide.isna().sum()
print(f'\nminute_intensities_wide:\n{na_counts.loc[na_counts > 0]}')

na_counts = minute_steps_narrow.isna().sum()
print(f'\nminute_steps_narrow:\n{na_counts.loc[na_counts > 0]}')

na_counts = minute_steps_wide.isna().sum()
print(f'\nminute_steps_wide:\n{na_counts.loc[na_counts > 0]}')

### Duplicate Rows

In [ ]:
# Duplicated rows
print('Duplicated rows')

print(f'minute_intensities_wide:\n{minute_intensities_wide.duplicated().sum()}')

print(f'\nminute_steps_narrow:\n{minute_steps_narrow.duplicated().sum()}')

print(f'\nminute_steps_wide:\n{minute_steps_wide.duplicated().sum()}')

### Observation

Data is very clean

### Intensity by Hour

In [ ]:
# Check minute_intensities_wide
DATE_FORMAT = '%m/%d/%Y %I:%M:%S %p'

minute_intensities_wide['time'] = pd.to_datetime(minute_intensities_wide['ActivityHour'], format=DATE_FORMAT).dt.hour

avg_intensities_per_hour = minute_intensities_wide.drop(columns='Id').groupby('time').mean(numeric_only=True).sum(axis=1)

fig, ax = plt.subplots()

ax.bar(avg_intensities_per_hour.index, avg_intensities_per_hour)
ax.set_title('Average Total Intensity by Hour of Day')
ax.set_xlabel('Hour of the Day (0-23)')
ax.set_ylabel('Average Intensity')

plt.show()

### Observation

Follows typical intensity throught the day

### Compare Steps Files

In [ ]:
# melt minute_steps_wide to narrow format to compare the data

minute_steps_melt = minute_steps_wide.melt(id_vars=['Id', 'ActivityHour'], var_name='minute', value_name='Steps')

minute_steps_melt['ActivityHour'] = pd.to_datetime(minute_steps_melt['ActivityHour'], format=DATE_FORMAT)
minute_steps_melt['minute'] = pd.to_timedelta(pd.to_numeric(minute_steps_melt['minute'].str[-2:]), unit='minutes')

minute_steps_melt['ActivityMinute'] = minute_steps_melt['ActivityHour'] + minute_steps_melt['minute']
minute_steps_melt = minute_steps_melt[['Id', 'ActivityMinute', 'Steps']]


minute_steps_narrow['ActivityMinute'] = pd.to_datetime(minute_steps_narrow['ActivityMinute'], format=DATE_FORMAT)

# Merge data
merged_steps = minute_steps_melt.merge(minute_steps_narrow, on=['Id', 'ActivityMinute'], how='outer', suffixes=('_wide', '_narrow'))

merged_steps['Steps'] = merged_steps['Steps_wide'].fillna(merged_steps['Steps_narrow'])

# Check overlapping data is correct
overlap = merged_steps.dropna(subset=['Steps_wide', 'Steps_narrow'])
print(f'Overlapping rows match: {(overlap["Steps_wide"] == overlap["Steps_narrow"]).all()}')

### Observation

Overlapping data is the same

### Steps by Hour

In [ ]:
# Plot graph
merged_steps['hour'] = merged_steps['ActivityMinute'].dt.hour
avg_steps_per_hour = merged_steps.groupby('hour')['Steps'].mean()

fig, ax = plt.subplots()

ax.bar(avg_steps_per_hour.index, avg_steps_per_hour)
ax.set_title('Average Steps per Minute by Hour of Day')
ax.set_xlabel('Hour of the Day (0-23)')
ax.set_ylabel('Average Steps per minute')

plt.show()

### Observation

Follows the intensity distribution and typical steps per hour of the day.

### Records and Participants

In [ ]:
for name, df in [('minute_intensities_wide', minute_intensities_wide),
                 ('minute_steps_narrow', minute_steps_narrow),
                 ('minute_steps_wide', minute_steps_wide)]:
    print(f'{name}: {len(df)} rows, {df["Id"].nunique()} participants')

### Observation

All three files cover the same 33 participants. The narrow file has more rows than the wide file melted (1,325,580 vs 21,645 × 60 = 1,298,700) so the two do not cover exactly the same minutes.

### Date and Time Coverage

In [ ]:
merged_steps['date'] = merged_steps['ActivityMinute'].dt.date

print(merged_steps['date'].agg(['min', 'max']))
print(merged_steps.groupby('Id')['date'].nunique().describe())

### Observation

Data spans 32 days (2016-04-12 to 2016-05-13). Most participants have 28-32 days of data, but at least one has only 4.

### Distributions and Outliers

In [ ]:
# Distribution of steps per minute and outliers in daily steps
daily_steps = merged_steps.groupby(['Id', 'date'])['Steps'].sum()

print(merged_steps['Steps'].describe())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
merged_steps['Steps'].plot.hist(bins=50, log=True, ax=axes[0], title='Steps per Minute')
daily_steps.plot.box(ax=axes[1], title='Daily Steps per Participant')

plt.show()

### Observation

Steps per minute is heavily skewed towards zero (median and 75th percentile are 0, max is 220). Daily steps have a few high outliers (max ~36,000 vs a median of ~7,300).